# Model Evaluation

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/01 22:53:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/01 22:53:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/01 22:53:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/08/01 22:53:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
from pyspark.ml.classification import DecisionTreeClassificationModel

model = DecisionTreeClassificationModel.load(
    "../models/decision_tree_model"
)

In [4]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

## Prepare Dataset

In [5]:
from pyspark.sql.functions import when, col
from pyspark.ml.feature import StringIndexer, VectorAssembler

df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1).otherwise(0)
)

indexer = StringIndexer(
    inputCol="operator",
    outputCol="operator_index"
)

df = indexer.fit(df).transform(df)

assembler = VectorAssembler(
    inputCols=[
        "number_of_stops",
        "number_of_journeys",
        "operator_index"
    ],
    outputCol="features"
)

dataset = assembler.transform(df)

train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

## Generate Predictions

In [6]:
predictions = model.transform(test_data)

## Model Accuracy

In [7]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy:", accuracy.evaluate(predictions))

[Stage 10:>                                                       (0 + 12) / 12]

Accuracy: 1.0


## Display Predictions

In [8]:
predictions.select(
    "route_size",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------+------+----------+-----------+
|route_size|target|prediction|probability|
+----------+------+----------+-----------+
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
+----------+------+----------+-----------+
only showing top 10 rows


## Prediction Distribution

In [9]:
predictions.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|   21|
|       1.0|    3|
+----------+-----+



## Summary

The Decision Tree model was successfully evaluated using the testing dataset. The model generated predictions for unseen data and achieved a measurable classification accuracy. The evaluation results indicate that the model can distinguish between long and non-long routes based on the selected features. These findings demonstrate the effectiveness of Spark MLlib for building scalable machine learning pipelines on timetable data.